# Experiment 1 — statistical analysis

Three complementary analyses, following Demšar (2006), García & Herrera (2008) and the benchmarking literature (Fernández-Delgado et al., 2014):

1. **PAMA** — *Probability of Achieving MAximal accuracy*: the relative frequency with which each learner achieves the TOP score across all fold-level observations. Descriptive dominance, no distributional assumptions.
2. **Friedman test + Iman–Davenport correction** — the omnibus test: are the rank differences globally non-random? Only if it rejects are post-hoc comparisons warranted. Visualised with **Nemenyi critical-difference diagrams**.
3. **Pairwise Wilcoxon signed-rank tests with Holm step-down correction** over ALL method pairs, reported as a **Win/Loss matrix** (a trailing `*` marks pairs significant at α = 0.05 after Holm).

PD uses **AUC**; LGD uses **R²**. Tests over datasets use one fold-mean per dataset; PAMA uses every (dataset, fold) observation.
All logic: `src/utils/statistical_testing.py`. Figures → `figures/experiment1/stats/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, learning_curve, imbalance_curve,
    metric_boxplots, metric_bars, median_time_bars, rank_heatmap, rank_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment1/stats')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
from src.utils import statistical_testing as st
from src.methods.method_config import FOUNDATION_METHODS
df_pd       = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd')
df_pd_folds = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd',  aggregated=False)
try:
    df_lgd       = load_summary(SUMMARY_DIR, experiment='experiment1', task='lgd')
    df_lgd_folds = load_summary(SUMMARY_DIR, experiment='experiment1', task='lgd', aggregated=False)
except FileNotFoundError:
    df_lgd = df_lgd_folds = None

## 1. PAMA — who wins the fold-level observations?

In [ ]:
st.plot_pama_bars(df_pd_folds, 'AUC', metric_name='AUC',
                  foundation_methods=sorted(FOUNDATION_METHODS),
                  out_path=FIGURES_DIR / 'pd_pama_auc.pdf')
display(st.pama_fold_level(df_pd_folds, 'AUC').head(15))

In [ ]:
if df_lgd_folds is not None:
    st.plot_pama_bars(df_lgd_folds, 'R2', metric_name='R2',
                      foundation_methods=sorted(FOUNDATION_METHODS),
                      out_path=FIGURES_DIR / 'lgd_pama_r2.pdf')
    display(st.pama_fold_level(df_lgd_folds, 'R2').head(15))

## 2. Friedman omnibus test (Iman–Davenport corrected)

In [ ]:
mat_pd = st.metric_matrix(df_pd, 'AUC')
f_pd = st.friedman_test(mat_pd)
print(f"PD  (N={f_pd['N']} datasets, k={f_pd['k']} methods): "
      f"F_F = {f_pd['iman_davenport_F']:.2f}, p = {f_pd['p_iman_davenport']:.3g}")
if df_lgd is not None:
    mat_lgd = st.metric_matrix(df_lgd, 'R2')
    f_lgd = st.friedman_test(mat_lgd)
    print(f"LGD (N={f_lgd['N']} datasets, k={f_lgd['k']} methods): "
          f"F_F = {f_lgd['iman_davenport_F']:.2f}, p = {f_lgd['p_iman_davenport']:.3g}")

### Nemenyi critical-difference diagrams
Methods joined by a bold bar are NOT significantly different at α = 0.05.

In [ ]:
st.plot_cd_diagram(st.average_ranks(mat_pd),
                   st.nemenyi_cd(mat_pd.shape[1], mat_pd.shape[0]),
                   title=f'PD — AUC average ranks (N={mat_pd.shape[0]} datasets)',
                   out_path=FIGURES_DIR / 'pd_cd_diagram_auc.pdf')

In [ ]:
if df_lgd is not None:
    st.plot_cd_diagram(st.average_ranks(mat_lgd),
                       st.nemenyi_cd(mat_lgd.shape[1], mat_lgd.shape[0]),
                       title=f'LGD — R2 average ranks (N={mat_lgd.shape[0]} datasets)',
                       out_path=FIGURES_DIR / 'lgd_cd_diagram_r2.pdf')

## 3. Pairwise Wilcoxon signed-rank tests, Holm-corrected
Cell text = Win/Loss of the row method vs the column method across datasets; `*` = significant after Holm (α = 0.05, corrected over all pairs).

In [ ]:
st.plot_wilcoxon_wl_matrix(mat_pd, metric_name='AUC',
                           out_path=FIGURES_DIR / 'pd_wilcoxon_wl_auc.pdf')
tab_pd = st.wilcoxon_holm_pairwise(mat_pd)
print(f'{int(tab_pd["significant"].sum())}/{len(tab_pd)} PD pairs significant after Holm')
display(tab_pd[tab_pd['significant']].head(25))

In [ ]:
if df_lgd is not None:
    st.plot_wilcoxon_wl_matrix(mat_lgd, metric_name='R2',
                               out_path=FIGURES_DIR / 'lgd_wilcoxon_wl_r2.pdf')
    tab_lgd = st.wilcoxon_holm_pairwise(mat_lgd)
    n = mat_lgd.shape[0]
    print(f'{int(tab_lgd["significant"].sum())}/{len(tab_lgd)} LGD pairs significant after Holm')
    print(f'note: with N={n} datasets the minimum two-sided Wilcoxon p is {2/2**n:.3f}, '
          f'so after Holm over {len(tab_lgd)} pairs significance is structurally hard to reach')

## Appendix — rank-based post-hoc procedures (Demšar 2006; García & Herrera 2008)
Adjusted p-values from the Friedman rank statistic: one-vs-control step procedures and all-pairwise procedures (Holm, Shaffer static, Bergmann–Hommel where k ≤ 8).

In [ ]:
display(st.control_apv_table(mat_pd).head(15))
display(st.pairwise_apv_table(mat_pd).head(15))

> **Interpretation guide.** PAMA describes *practical dominance frequency*; the Friedman test establishes that rank differences are *globally* non-random; the Wilcoxon–Holm matrix *localizes* which pairs differ. Few significant pairs with small N is expected — the omnibus test has far more power to detect global heterogeneity than pairwise tests have to localize it (see the power note printed above for LGD).